In [27]:
!git clone https://github.com/karpathy/minbpe.git
import sys
sys.path.append('/content/minbpe')
from minbpe.base import Tokenizer, get_stats, merge

fatal: destination path 'minbpe' already exists and is not an empty directory.


In [28]:
"""
Exercise: Minimal (byte-level) Byte Pair Encoding tokenizer.
Objective: Fill in the missing logic for 'train' and 'encode'.
The tokenizer should convert text to UTF-8 bytes, then iteratively
merge the most frequent adjacent pairs.
https://youtu.be/zduSFxRajkE?t=1433 (watch to help)
"""
from minbpe.base import Tokenizer, get_stats, merge

class BasicTokenizer(Tokenizer):
    def __init__(self):
        super().__init__()

    def train(self, text, vocab_size, verbose=False):
        """
        Trains the tokenizer by finding the most frequent byte pairs and
        merging them into new tokens until vocab_size is reached.
        """
        assert vocab_size >= 256, "Vocab size must at least cover all single bytes (0-255)"
        num_merges = vocab_size - 256

        # Step 1: Convert the input string into a list of raw byte integers (0-255)
        # YOUR CODE HERE: Initialize 'ids'
        ids = list(text.encode("utf-8"))

        # Step 2: Initialize the vocabulary
        # Each index 0-255 should map to its corresponding byte object
        merges = {} # (int, int) -> int
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            # Step 3: Count frequencies of all consecutive pairs in 'ids'
            # Hint: Use the provided get_stats(ids) helper
            # YOUR CODE HERE
            stats = get_stats(ids)

            # Step 4: Identify the pair with the highest frequency
            # If there are no pairs or stats is empty, break early
            # YOUR CODE HERE
            if not stats:
                break
            pair = max(stats, key=stats.get)

            # Step 5: "Mint" a new token ID
            # The first new token should be 256, then 257, etc.
            # YOUR CODE HERE
            idx = 256 + i

            # Step 6: Update 'ids' by replacing all occurrences of the chosen 'pair' with 'idx'
            # Hint: Use the provided merge(ids, pair, idx) helper
            # YOUR CODE HERE
            ids = merge(ids, pair, idx)

            # Step 7: Document the merge in the 'merges' and 'vocab' dictionaries
            # 'merges' maps the pair (tuple) to the new idx
            # 'vocab' maps the new idx to the concatenated bytes of the two components
            # YOUR CODE HERE
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]})")

        # Save results to the class
        self.merges = merges
        self.vocab = vocab

    def decode(self, ids):
        """
        Converts a list of token IDs back into a Python string.
        """
        # Join the bytes associated with each ID in the vocab
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        # Decode bytes to string, replacing errors for robustness
        return text_bytes.decode("utf-8", errors="replace")

    def encode(self, text):
        """
        Converts a string into a list of token IDs using the learned merges.
        """
        # Step 1: Start with the raw byte list
        # YOUR CODE HERE
        ids = list(text.encode("utf-8"))

        # Step 2: Iteratively apply merges
        while len(ids) >= 2:
            # Step 3: Find all current pair frequencies
            # YOUR CODE HERE
            stats = get_stats(ids)

            # Step 4: Find the pair that was merged FIRST during training.
            # BPE must apply merges in the exact order they were learned.
            # Hint: Look for the pair in 'self.merges' with the minimum value (ID).
            # If no pairs in 'ids' are found in 'self.merges', we are done.
            # YOUR CODE HERE
            pair = min(
                (p for p in stats if p in self.merges),
                key=lambda p: self.merges[p],
                default=None,
            )

            # Step 5: Termination condition
            # YOUR CODE HERE: if pair not in self.merges, break
            if pair is None:
                break

            # Step 6: Replace the pair in our 'ids' list and repeat
            # YOUR CODE HERE
            ids = merge(ids, pair, self.merges[pair])

        return ids


In [29]:
# encode
ids = tokenizer.encode("hello world")
print("Encoded:", ids)

# decode
decoded = tokenizer.decode(ids)
print("Decoded:", decoded)

Encoded: [257, 108, 111, 32, 119, 111, 114, 108, 100]
Decoded: hello world


In [30]:
ids = tokenizer.encode("مرحبا بالعالم")
print("Encoded:", ids)
print("Decoded:", tokenizer.decode(ids))

Encoded: [217, 133, 216, 177, 216, 173, 216, 168, 216, 167, 32, 216, 168, 216, 167, 217, 132, 216, 185, 216, 167, 217, 132, 217, 133]
Decoded: مرحبا بالعالم
